# EX_06 — Introducción a RAG (ejercicios)

**Notebook de referencia:** `notebook/06_Introduccion_RAG.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Plantilla de contexto

Escribe una función `build_prompt(context_chunks, question) -> str` que inserte los pasajes en un delimitador claro (`### Context` / `### Question`).


In [2]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    # TODO
    """
    Construye un prompt estructurado inyectando fragmentos de contexto 
    y una pregunta específica para un sistema RAG.
    """
    
    # 1. Unimos los fragmentos de contexto con saltos de línea y numeración
    # para que el modelo pueda referenciarlos si es necesario.
    context_str = "\n\n".join([f"Passage {i+1}: {chunk.strip()}" for i, chunk in enumerate(context_chunks)])
    
    # 2. Definimos la plantilla con delimitadores claros
    template = f"""
Usa ÚNICAMENTE la siguiente información para responder a la pregunta. 
Si la respuesta no está en el contexto, indica que no tienes suficiente información.

### Context
{context_str}

### Question
{question}

### Answer
"""
    return template.strip()

# --- Ejemplo de uso ---
chunks = [
    "La capital de Marte es una base hipotética llamada Chryse Planitia.",
    "El monte Olimpo en Marte es el volcán más grande del sistema solar."
]
pregunta = "¿Cuál es la capital de Marte según el contexto?"

prompt_final = build_prompt(chunks, pregunta)
print(prompt_final)


Usa ÚNICAMENTE la siguiente información para responder a la pregunta. 
Si la respuesta no está en el contexto, indica que no tienes suficiente información.

### Context
Passage 1: La capital de Marte es una base hipotética llamada Chryse Planitia.

Passage 2: El monte Olimpo en Marte es el volcán más grande del sistema solar.

### Question
¿Cuál es la capital de Marte según el contexto?

### Answer


## Actividad 2 — RAG sin LLM (retrieval only)

Con tus chunks del notebook teórico (o texto inventado), recupera top-k y **imprime** el contexto ensamblado sin llamar al generador.


In [1]:
# TODO: retrieve top-k for a fixed question
from sentence_transformers import SentenceTransformer, util

# 1. Tu base de conocimiento (Chunks indexados)
knowledge_base = [
    "Los gatos domésticos pasan cerca del 70% del día durmiendo para conservar energía.",
    "El telescopio James Webb opera principalmente en el espectro infrarrojo.",
    "La fotosíntesis es el proceso mediante el cual las plantas convierten luz solar en glucosa.",
    "Los felinos tienen una excelente visión nocturna debido a una estructura llamada tapetum lucidum.",
    "El ciclo de Krebs ocurre dentro de las mitocondrias de las células eucariotas."
]

# 2. Inicializar el modelo de embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(knowledge_base)

# 3. Definir la consulta del usuario
query = "¿Por qué los gatos duermen tanto tiempo y cómo ven de noche?"
query_embedding = model.encode(query)

# 4. Fase de Recuperación (Retrieval - Top-k)
k = 2
scores = util.cos_sim(query_embedding, kb_embeddings)[0].tolist()

# Emparejamos texto con score y ordenamos
ranked_chunks = sorted(zip(knowledge_base, scores), key=lambda x: x[1], reverse=True)
top_k_chunks = [doc for doc, score in ranked_chunks[:k]]

# 5. Fase de Ensamblaje (Prompt Engineering)
def build_rag_prompt(context_chunks: list[str], question: str) -> str:
    context_str = "\n".join([f"[- ] {chunk}" for chunk in context_chunks])
    prompt = f"""=== CONTEXTO DE REFERENCIA ===
{context_str}

=== INSTRUCCIÓN ===
Responde a la pregunta utilizando únicamente el contexto proporcionado arriba.

Pregunta: {question}
Respuesta:"""
    return prompt

# 6. Imprimir el resultado final que se enviaría al LLM
final_prompt = build_rag_prompt(top_k_chunks, query)

print(final_prompt)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== CONTEXTO DE REFERENCIA ===
[- ] Los gatos domésticos pasan cerca del 70% del día durmiendo para conservar energía.
[- ] Los felinos tienen una excelente visión nocturna debido a una estructura llamada tapetum lucidum.

=== INSTRUCCIÓN ===
Responde a la pregunta utilizando únicamente el contexto proporcionado arriba.

Pregunta: ¿Por qué los gatos duermen tanto tiempo y cómo ven de noche?
Respuesta:


## Actividad 3 — Fallo de cobertura

Inventa un caso donde la respuesta **no** está en los chunks recuperados y describe en español (markdown) cómo lo detectarías en producción (p. ej. umbral de score, abstención).


_Tu explicación:_

1. El Caso

Un usuario pregunta: "¿Cómo exporto mis facturas mensuales a PDF automáticamente?" El sistema recupera fragmentos que hablan de "métodos de pago" y "facturas por correo HTML". El buscador vectorial los selecciona porque comparten palabras clave ("factura", "mensual"), pero ninguno contiene la solución específica (exportar a PDF).

2. Detección en Producción (4 Capas de Control)

Umbral de Score:
Se establece un límite mínimo de similitud coseno para el bloque principal. Si el valor del fragmento recuperado no supera este número crítico, el sistema detiene el proceso de inmediato y evita enviar información inservible al generador.

Validación con Reranker:
Un modelo Cross-Encoder analiza a fondo la relación entre la consulta y los textos recuperados. Al notar que el contenido no resuelve la intención real del usuario, descarta el fragmento aunque comparta palabras idénticas.

Instrucción de Abstención:
El prompt del sistema obliga al modelo a responder con una frase predefinida, como "No dispongo de esa información", en caso de que la respuesta exacta no aparezca de forma explícita en los textos de referencia.

Evaluación Post-Salida:
Un evaluador automatizado contrasta la respuesta final del modelo con el contexto original. Si la salida detalla pasos para generar un PDF que nunca estuvieron en los textos de origen, el sistema bloquea el mensaje antes de que llegue al usuario para evitar una alucinación.
